In [ ]:
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import time
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import random
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn.functional as F

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Download dataset
itsahmad_indoor_scenes_cvpr_2019_path = kagglehub.dataset_download('itsahmad/indoor-scenes-cvpr-2019')
print('Data source import complete.')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Setup paths
basePath = itsahmad_indoor_scenes_cvpr_2019_path
imgDir = os.path.join(basePath, "indoorCVPR_09", "Images")
trainLabelFile = os.path.join(basePath, "TrainImages.txt")
testLabelFile = os.path.join(basePath, "TestImages.txt")

# Enhanced train/validation split with stratification
def create_train_val_split(train_file, val_ratio=0.15):  # Reduced val ratio for more training data
    """Create a stratified train/validation split"""
    with open(train_file, 'r') as f:
        train_lines = f.read().splitlines()
    
    class_to_images = defaultdict(list)
    for line in train_lines:
        if line.strip():
            class_name = line.split('/')[0]
            class_to_images[class_name].append(line)
    
    train_list, val_list = [], []
    for cls, images in class_to_images.items():
        random.shuffle(images)
        split_idx = max(1, int((1 - val_ratio) * len(images)))  # Ensure at least 1 image per class in val
        train_list += images[:split_idx]
        val_list += images[split_idx:]
    
    return train_list, val_list

# Create proper split
train_list, val_list = create_train_val_split(trainLabelFile, val_ratio=0.15)
print(f"Train: {len(train_list)} images, Val: {len(val_list)} images")

class Indoor67Dataset(Dataset):
    def __init__(self, imgDir, image_list, transform=None):
        self.samples = []
        self.transform = transform
        
        for imgName in image_list:
            if imgName.strip():
                label = imgName.split('/')[0]
                fullPath = os.path.join(imgDir, imgName)
                if os.path.isfile(fullPath):
                    self.samples.append((fullPath, label))
        
        self.classes = sorted(set(lbl for _, lbl in self.samples))
        self.classToIdx = {cls: i for i, cls in enumerate(self.classes)}
        print(f"Loaded {len(self.samples)} samples with {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            return image, self.classToIdx[label]
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            # Return a dummy image if loading fails
            dummy_image = torch.zeros(3, 224, 224)
            return dummy_image, self.classToIdx[label]

# Enhanced data augmentation
trainTransform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(0.8, 1.2)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.RandomApply([transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15), ratio=(0.3, 3.3))  # Cutout augmentation
])

# Test Time Augmentation transforms
valTransform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

testTransform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create datasets
trainDataset = Indoor67Dataset(imgDir, train_list, transform=trainTransform)
valDataset = Indoor67Dataset(imgDir, val_list, transform=valTransform)

# For test, read from file
with open(testLabelFile, 'r') as f:
    test_list = f.read().splitlines()
testDataset = Indoor67Dataset(imgDir, test_list, transform=testTransform)

# Optimized data loading
trainLoader = DataLoader(trainDataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
valLoader = DataLoader(valDataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
testLoader = DataLoader(testDataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# Enhanced model with multiple architectures ensemble capability
class EnhancedModel(nn.Module):
    def __init__(self, num_classes, model_name='efficientnet_b3'):
        super(EnhancedModel, self).__init__()
        
        if model_name == 'efficientnet_b3':
            self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
            num_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()
        elif model_name == 'resnet152':
            self.backbone = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:  # default resnet50
            self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        
        # Enhanced classifier with multiple dropout layers
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(num_features),
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# Try EfficientNet-B3 for better performance
model = EnhancedModel(len(trainDataset.classes), model_name='efficientnet_b3')
model = model.to(device)

# Enhanced loss with focal loss for handling class imbalance
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(alpha=1, gamma=2, label_smoothing=0.1)

# Advanced optimizer with different learning rates for backbone and classifier
backbone_params = list(model.backbone.parameters())
classifier_params = list(model.classifier.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 0.0001},  # Lower LR for pretrained backbone
    {'params': classifier_params, 'lr': 0.001}  # Higher LR for new classifier
], weight_decay=0.01)

# Cosine annealing with warm restarts
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Mixup augmentation
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_model(model, train_loader, val_loader, num_epochs=80, patience=15):
    train_losses, train_accs, val_losses, val_accs = [], [], [], []
    best_val_acc = 0.0
    patience_counter = 0
    
    # Gradient scaler for mixed precision training
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        
        for batch_idx, (images, labels) in enumerate(progress_bar):
            images, labels = images.to(device), labels.to(device)
            
            # Apply mixup with probability 0.5
            if random.random() > 0.5:
                images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.2)
                
                optimizer.zero_grad()
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        outputs = model(images)
                        loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(images)
                    loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                
                # For accuracy calculation with mixup
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (lam * predicted.eq(labels_a).sum().float() + 
                           (1 - lam) * predicted.eq(labels_b).sum().float()).item()
            else:
                optimizer.zero_grad()
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'Loss': f'{running_loss/(batch_idx+1):.4f}', 
                                    'Acc': f'{100.*correct/total:.2f}%'})
        
        scheduler.step()
        
        train_acc = correct / total
        train_loss = running_loss / len(train_loader)
        train_accs.append(train_acc)
        train_losses.append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                if scaler:
                    with torch.cuda.amp.autocast():
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = val_correct / val_total
        val_loss = val_loss / len(val_loader)
        val_accs.append(val_acc)
        val_losses.append(val_loss)
        
        print(f'Epoch [{epoch+1}/{num_epochs}] - '
              f'Train Acc: {train_acc:.4f}, Loss: {train_loss:.4f} | '
              f'Val Acc: {val_acc:.4f}, Loss: {val_loss:.4f} | '
              f'LR: {optimizer.param_groups[0]["lr"]:.6f}')
        
        # Early stopping with model saving
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'epoch': epoch
            }, 'best_model_enhanced.pth')
            print(f"New best model saved with val acc: {val_acc:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    return train_losses, train_accs, val_losses, val_accs

# Test Time Augmentation
def test_time_augmentation(model, image, num_augmentations=5):
    """Apply test time augmentation for better predictions"""
    augment_transforms = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    predictions = []
    
    # Original prediction
    with torch.no_grad():
        pred = model(image.unsqueeze(0))
        predictions.append(F.softmax(pred, dim=1))
    
    # Augmented predictions
    for _ in range(num_augmentations):
        # Convert back to PIL for augmentation
        pil_image = transforms.ToPILImage()(image.cpu())
        aug_image = augment_transforms(pil_image).to(device)
        
        with torch.no_grad():
            pred = model(aug_image.unsqueeze(0))
            predictions.append(F.softmax(pred, dim=1))
    
    # Average all predictions
    avg_pred = torch.mean(torch.stack(predictions), dim=0)
    return avg_pred

# Enhanced evaluation with TTA
def evaluate_model_with_tta(model, test_loader, use_tta=True):
    model.eval()
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            
            if use_tta and images.size(0) == 1:  # TTA for single images
                outputs = test_time_augmentation(model, images[0])
            else:
                outputs = model(images)
                outputs = F.softmax(outputs, dim=1)
            
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = correct / total
    return accuracy, all_predictions, all_labels
epochs=1
print("Starting enhanced training...")
train_losses, train_accs, val_losses, val_accs = train_model(
    model, trainLoader, valLoader, num_epochs=epochs, patience=20
)

# Load best model for testing
checkpoint = torch.load('best_model_enhanced.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} with val acc: {checkpoint['val_acc']:.4f}")

# Enhanced evaluation
test_accuracy, predictions, true_labels = evaluate_model_with_tta(model, testLoader, use_tta=True)
print(f'Test Accuracy with TTA: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

# Regular evaluation without TTA for comparison
test_accuracy_no_tta, _, _ = evaluate_model_with_tta(model, testLoader, use_tta=False)
print(f'Test Accuracy without TTA: {test_accuracy_no_tta:.4f} ({test_accuracy_no_tta*100:.2f}%)')

# Detailed classification report
class_names = testDataset.classes
print("\nDetailed Classification Report:")
print(classification_report(true_labels, predictions, target_names=class_names, digits=4))

# Plot enhanced training history
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
plt.plot(train_accs, label='Train Acc', linewidth=2)
plt.plot(val_accs, label='Val Acc', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 3)
plt.plot(np.gradient(val_accs), label='Val Acc Gradient', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy Gradient')
plt.legend()
plt.title('Validation Accuracy Improvement Rate')
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
epochs = range(len(val_accs))
plt.plot(epochs, [max(val_accs[:i+1]) for i in epochs], label='Best Val Acc So Far', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Best Accuracy')
plt.legend()
plt.title('Best Validation Accuracy Progress')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"Best validation accuracy: {max(val_accs):.4f} ({max(val_accs)*100:.2f}%)")
print(f"Test accuracy (no TTA): {test_accuracy_no_tta:.4f} ({test_accuracy_no_tta*100:.2f}%)")
print(f"Test accuracy (with TTA): {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Improvement with TTA: {(test_accuracy - test_accuracy_no_tta)*100:.2f}%")